In [29]:
# Install ADK
!pip install google-adk requests

# Authenticate your notebook
from google.colab import auth
auth.authenticate_user()

# Configure environment variables
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-04-3d3bb8d72ee3"
os.environ["GOOGLE_CLOUD_REGION"] = "us-central1"

# Enable the required enterprise Agent Engine APIs
!gcloud services enable aiplatform.googleapis.com

In [15]:
!mkdir -p enterprise_weather_agent
%cd enterprise_weather_agent

/content/weather_agent_project/enterprise_weather_agent


In [16]:
%%writefile agent.py
import requests
from google.adk.agents import Agent
from google.adk.tools import tool

@tool
def get_nws_forecast(latitude: float, longitude: float) -> str:
    """Fetches the weather forecast from the National Weather Service (NWS) API."""
    headers = {"User-Agent": "(MyEnterpriseAgent, contact@example.com)"}
    try:
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        points_res = requests.get(points_url, headers=headers).json()
        forecast_url = points_res["properties"]["forecast"]

        forecast_res = requests.get(forecast_url, headers=headers).json()
        periods = forecast_res["properties"]["periods"]

        return "\n".join([f"{p['name']}: {p['temperature']}°F, {p['shortForecast']}." for p in periods[:3]])
    except Exception as e:
        return f"Error: {str(e)}"

# Define the root agent variable matching platform requirements
root_agent = Agent(
    name="nws-weather-agent",
    model="gemini-2.0-flash",
    instruction="You are an enterprise weather agent. Call your forecast tool with coordinates to answer queries.",
    tools=[get_nws_forecast]
)

Writing agent.py


In [17]:
%%writefile __init__.py
from . import agent

Writing __init__.py


In [18]:
%%writefile requirements.txt
google-adk
requests

Writing requirements.txt


In [30]:

# Deploy explicitly to Agent Platform (Agent Engine)
!adk deploy agent_engine \
    --project=$GOOGLE_CLOUD_PROJECT \
    --region=$GOOGLE_CLOUD_REGION \
    .

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
Agent Engine deployment uses relative paths; temporarily switching working directory to: /content/weather_agent_project
Staging all files in: /content/weather_agent_project/enterprise_weather_agent_tmp20260709_212932
Staging all files in: /content/weather_agent_project/enterprise_weather_agent_tmp20260709_212932
Copying agent source code...
Copying agent source code complete.
Resolving files and dependencies...
Initializing Vertex AI...
Vertex AI initialized.
Created enterprise_weather_agent_tmp20260709_212932/agent_engine_app.py
Files and dependencies resolved
Deploying to agent engine...
Cleaning up the temp folder: enterprise_weather_agent_tmp20260709_212932
Deploy failed: Failed to create Agent Engine: {'code': 3, 'message': 'Reasoning Engine resource [projects/153302561600/locations/us-central1/re

In [ ]:
!adk chat \
    --agent nws-weather-agent \
    --message "What is the forecast around coordinates 34.0522, -118.2437?"